In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [9]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

In [10]:
len(documents)

72

In [11]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing and searching

In [12]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [13]:
question = "How does the agentic loop keep calling the model until it stops?"

In [14]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. RAG

In [15]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper1 import RAGBase

from openai import OpenAI
openai_client = OpenAI()

In [16]:
assistant = RAGBase(index, openai_client)

In [17]:
response =assistant.rag(question)

In [18]:
response

{'answer': Response(id='resp_0870175ca940d3a9006a39cf3da1908193ab13e06f16da031e', created_at=1782173501.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0870175ca940d3a9006a39cf3e41308193982544b5289584a1', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop.\n\nEach turn it:\n1. sends the current `messages` to the model,\n2. checks the response for any `function_call`s,\n3. runs those tools and appends the outputs to `messages`,\n4. repeats.\n\nIt stops when a model response contains no function calls:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the model decides when to stop by returning a final answer without asking for more tools.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice

## Q4. Chunking

In [19]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
len(chunks)

295

## Q5. RAG with chunking

In [21]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [22]:
assistant1 = RAGBase(chunk_index, openai_client)

In [23]:
response1 = assistant1.rag(question)

In [24]:
response1

{'answer': Response(id='resp_065d49687f65c2c8006a39cf4002408197b12ae68dc6fccb5b', created_at=1782173504.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_065d49687f65c2c8006a39cf40a84481979ac85030ca61f2f4', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model with `while True`, then checks whether the response included any `function_call` items.\n\n- If there **are** function calls, the code runs them, appends the tool outputs to `messages`, and loops again.\n- If there are **no** function calls, it breaks out of the loop and returns the final answer.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, it keeps going until the model returns a response with no more tool calls.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], 

In [25]:
response['prompt_tokens'] / response1["prompt_tokens"]

3.0771884432945233

## Q6. Turning it into an agent

In [26]:
!uv add toyaikit

Resolved 129 packages in 2ms
Checked 125 packages in 13ms


In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

In [29]:
search_calls = []

def search(query: str) -> str:
    """
    Search the course materials.

    Args:
        query: Search query.

    Returns:
        Relevant chunks from the course materials.
    """
    search_calls.append(query)

    results = chunk_index.search(query, num_results=5)

    return "\n\n".join(
        doc["content"]
        for doc in results
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [32]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [33]:
len(search_calls)

3

In [34]:
search_calls

['agentic loop RAG course materials',
 'plain RAG vs agentic loop retrieval augmented generation',
 'agentic workflow loop plan act observe course']